# Mammography — primary model training

Training and evaluation of U-Net, Attention U-Net, and Swin-Tiny-U-Net across the predefined random seeds using CBIS-DDSM development data and the held-out INbreast cohort.


In [ ]:
from pathlib import Path
import numpy as np

# Leave this path as None for automatic discovery under /kaggle/input.
# Accepts either ROI_Crops_256_v1_Kaggle.zip or an already extracted folder containing roi_crop_manifest.csv.
SOURCE_PATH = None
# SOURCE_PATH = "/kaggle/input/roi-crops-256-v1/ROI_Crops_256_v1_Kaggle.zip"

OUT_DIR = Path("/kaggle/working/ROI256_TRAINING_RESULTS")

# Fast smoke-test: use ["unet"] and RUN_FAST_DEV=True.
# Full benchmark: ["unet", "attention_unet", "swin_tiny_unet"]
MODELS_TO_RUN = ["unet", "attention_unet", "swin_tiny_unet"]
TRAINING_SEEDS = [42, 123, 2025]

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 100
PATIENCE = 15
AMP = True
RUN_FAST_DEV = False

BASE_LR = 2e-4
ENCODER_LR = 2e-5
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRAD_CLIP_NORM = 1.0

THRESHOLDS = [round(float(x), 3) for x in np.arange(0.05, 0.96, 0.05)]

AUGMENT_TRAIN = True
HFLIP_P = 0.5
ROT_DEG = 7.0
TRANSLATE_FRAC = 0.03
SCALE_LOW = 0.95
SCALE_HIGH = 1.05

SAVE_QUALITATIVE_N = 16
print("Configuration loaded successfully.")

In [ ]:
import os, sys, json, math, time, random, shutil, zipfile, hashlib, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display, Image as IPyImage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    MATPLOTLIB_AVAILABLE = False
    print("matplotlib is unavailable:", repr(e))

try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    print("scipy is unavailable: HD95/ASD will be NaN", repr(e))

try:
    import torchvision
    import torchvision.transforms.functional as TF
    from torchvision.transforms import InterpolationMode
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_AVAILABLE = False
    print("torchvision is unavailable: Swin/advanced augmentations are unavailable", repr(e))

OUT_DIR.mkdir(parents=True, exist_ok=True)
for d in ["checkpoints", "metrics", "figures", "logs"]:
    (OUT_DIR / d).mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torchvision:", getattr(torchvision, "__version__", "NA") if TORCHVISION_AVAILABLE else "NA")
print("SciPy:", SCIPY_AVAILABLE)

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
def find_manifest_in_folder(folder: Path):
    hits = list(Path(folder).rglob("roi_crop_manifest.csv"))
    if not hits:
        return None
    return sorted(hits, key=lambda p: len(p.parts))[0]

def zip_contains_manifest(zip_path: Path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            return any(Path(n).name == "roi_crop_manifest.csv" for n in z.namelist())
    except Exception:
        return False

def print_input_tree(max_items=160):
    root = Path("/kaggle/input")
    print("Preview of /kaggle/input")
    if not root.exists():
        print("  /kaggle/input does not exist")
        return
    items = list(root.rglob("*"))
    for p in items[:max_items]:
        print(" ", p)
    if len(items) > max_items:
        print(" ...", len(items) - max_items, "additional items")

def resolve_source_path(source_path=None):
    if source_path is not None:
        p = Path(source_path)
        if not p.exists():
            raise FileNotFoundError(f"SOURCE_PATH introuvable: {p}")
        return p

    input_root = Path("/kaggle/input")
    manifests = list(input_root.rglob("roi_crop_manifest.csv"))
    if manifests:
        return sorted(manifests, key=lambda p: len(p.parts))[0].parent

    zips = sorted(input_root.rglob("*.zip"))
    print("ZIP files found:", len(zips))
    for z in zips:
        print(" -", z)
    candidates = [z for z in zips if zip_contains_manifest(z)]
    if candidates:
        return candidates[0]

    print_input_tree()
    raise FileNotFoundError(
        "No ROI_Crops_256 ZIP or folder containing roi_crop_manifest.csv was found. "
        "Please add ROI_Crops_256_v1_Kaggle.zip using Kaggle Add Data."
    )

def prepare_dataset_root(source_path):
    source_path = Path(source_path)
    if source_path.is_dir():
        manifest_path = find_manifest_in_folder(source_path)
        if manifest_path is None:
            raise FileNotFoundError(f"roi_crop_manifest.csv introuvable dans {source_path}")
        return manifest_path.parent

    if source_path.suffix.lower() == ".zip":
        extract_dir = Path("/kaggle/working/ROI_Crops_256_v1_extracted")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        print("Extracting:", source_path)
        with zipfile.ZipFile(source_path, "r") as z:
            bad = z.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupted ZIP member: {bad}")
            z.extractall(extract_dir)
        manifest_path = find_manifest_in_folder(extract_dir)
        if manifest_path is None:
            raise FileNotFoundError("roi_crop_manifest.csv not found after extraction")
        return manifest_path.parent

    raise ValueError(f"Unsupported source: {source_path}")

SOURCE = resolve_source_path(SOURCE_PATH)
DATA_ROOT = prepare_dataset_root(SOURCE)
manifest = pd.read_csv(DATA_ROOT / "roi_crop_manifest.csv")

print("SOURCE:", SOURCE)
print("DATA_ROOT:", DATA_ROOT)
print("Manifest:", manifest.shape)
display(manifest.head())

audit = {"source": str(SOURCE), "data_root": str(DATA_ROOT)}
if SOURCE.is_file():
    audit["source_sha256"] = sha256_file(SOURCE)
pd.DataFrame([audit]).to_csv(OUT_DIR / "logs" / "roi256_source_audit.csv", index=False)

In [ ]:
required_cols = [
    "sample_id", "dataset", "source", "split", "patient_id", "case_id", "laterality",
    "oracle_crop_flag", "lesion_ratio_original", "lesion_ratio_crop",
    "bbox_w", "bbox_h", "crop_size_native", "crop_touches_border", "npz_path"
]
missing = [c for c in required_cols if c not in manifest.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

display(manifest.groupby(["dataset", "split"]).size().reset_index(name="n"))

cbis = manifest[manifest["dataset"].astype(str).str.upper().str.contains("CBIS")].copy()
inb = manifest[manifest["dataset"].astype(str).str.upper().str.contains("INBREAST")].copy()

if cbis.empty:
    raise RuntimeError("No CBIS-DDSM sample found in the manifest")
if not {"train", "validation", "test"}.issubset(set(cbis["split"].astype(str))):
    raise RuntimeError("CBIS-DDSM must contain train/validation/test splits")

if not inb.empty and not set(inb["split"].astype(str)).issubset({"external_inbreast"}):
    raise RuntimeError("INbreast must be assigned only to external_inbreast")

split_patients = {
    s: set(cbis.loc[cbis["split"].astype(str).eq(s), "patient_id"].astype(str))
    for s in ["train", "validation", "test"]
}
for a in split_patients:
    for b in split_patients:
        if a < b:
            inter = split_patients[a] & split_patients[b]
            if inter:
                raise RuntimeError(f"CBIS patient leakage between {a} and {b}: {len(inter)} patients")
print("CBIS patient-level leakage check: PASSED")

missing_npz = []
for rel in manifest["npz_path"].astype(str):
    if not (DATA_ROOT / rel).exists():
        missing_npz.append(rel)
if missing_npz:
    raise FileNotFoundError(f"Missing NPZ files: {len(missing_npz)} examples={missing_npz[:5]}")
print("Manifest NPZ files: PASSED")

In [ ]:
class ROI256Dataset(Dataset):
    def __init__(self, df, root, augment=False, image_size=256):
        self.df = df.reset_index(drop=True).copy()
        self.root = Path(root)
        self.augment = augment
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with np.load(self.root / row["npz_path"], allow_pickle=False) as z:
            image = z["image"].astype(np.float32)
            mask = z["mask"].astype(np.uint8)

        image = np.squeeze(image)
        mask = np.squeeze(mask)
        image = np.clip(image, 0, 1).astype(np.float32)
        mask = (mask > 0).astype(np.float32)

        image_t = torch.from_numpy(image).unsqueeze(0).float()
        mask_t = torch.from_numpy(mask).unsqueeze(0).float()

        if self.augment and TORCHVISION_AVAILABLE:
            image_t, mask_t = self.apply_augmentation(image_t, mask_t)

        return {
            "image": image_t,
            "mask": mask_t,
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "dataset": str(row["dataset"]),
            "split": str(row["split"]),
        }

    def apply_augmentation(self, image_t, mask_t):
        if random.random() < HFLIP_P:
            image_t = TF.hflip(image_t)
            mask_t = TF.hflip(mask_t)

        angle = random.uniform(-ROT_DEG, ROT_DEG)
        max_t = int(TRANSLATE_FRAC * self.image_size)
        translate = (random.randint(-max_t, max_t), random.randint(-max_t, max_t))
        scale = random.uniform(SCALE_LOW, SCALE_HIGH)

        image_t = TF.affine(
            image_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR, fill=0.0
        )
        mask_t = TF.affine(
            mask_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0],
            interpolation=InterpolationMode.NEAREST, fill=0.0
        )
        mask_t = (mask_t > 0.5).float()
        return image_t, mask_t

def make_loaders(seed):
    g = torch.Generator()
    g.manual_seed(seed)

    train_df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("train"))].copy()
    val_df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("validation"))].copy()
    test_df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("test"))].copy()
    ext_df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("INBREAST")) & (manifest["split"].eq("external_inbreast"))].copy()

    if RUN_FAST_DEV:
        train_df = train_df.sample(min(len(train_df), 64), random_state=seed)
        val_df = val_df.sample(min(len(val_df), 32), random_state=seed)
        test_df = test_df.sample(min(len(test_df), 32), random_state=seed)
        if len(ext_df):
            ext_df = ext_df.sample(min(len(ext_df), 32), random_state=seed)

    loaders = {
        "train": DataLoader(
            ROI256Dataset(train_df, DATA_ROOT, augment=AUGMENT_TRAIN, image_size=IMAGE_SIZE),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
        "train_eval": DataLoader(
            ROI256Dataset(train_df, DATA_ROOT, augment=False, image_size=IMAGE_SIZE),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
    }
    for split, df in [("validation", val_df), ("test", test_df), ("external_inbreast", ext_df)]:
        loaders[split] = DataLoader(
            ROI256Dataset(df, DATA_ROOT, augment=False, image_size=IMAGE_SIZE),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        )
    return loaders

loaders_test = make_loaders(42)
for k, v in loaders_test.items():
    print(k, len(v.dataset), "samples")
b = next(iter(loaders_test["train"]))
print("Sanity-check batch:", b["image"].shape, b["mask"].shape, b["image"].min().item(), b["image"].max().item())

In [ ]:
def make_ground_truth_panel(split_name, n=8):
    """Save a visual QC panel with input ROI and ground-truth mask overlay for a dataset split."""
    if not MATPLOTLIB_AVAILABLE:
        return None
    if split_name == "train":
        df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("train"))].copy()
    elif split_name == "validation":
        df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("validation"))].copy()
    elif split_name == "test":
        df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("CBIS")) & (manifest["split"].eq("test"))].copy()
    elif split_name == "external_inbreast":
        df = manifest[(manifest["dataset"].astype(str).str.upper().str.contains("INBREAST")) & (manifest["split"].eq("external_inbreast"))].copy()
    else:
        raise ValueError(split_name)

    if df.empty:
        print(f"No samples available for split: {split_name}")
        return None

    df = df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)
    cols = 4
    rows = int(math.ceil(len(df) / cols))
    plt.figure(figsize=(cols * 4, rows * 4))
    for j, row in df.iterrows():
        with np.load(DATA_ROOT / row["npz_path"], allow_pickle=False) as z:
            img = np.squeeze(z["image"]).astype(np.float32)
            mask = (np.squeeze(z["mask"]) > 0).astype(np.uint8)
        rgb = np.stack([img, img, img], axis=-1)
        rgb[..., 1] = np.maximum(rgb[..., 1], mask * 0.95)
        ax = plt.subplot(rows, cols, j + 1)
        ax.imshow(np.clip(rgb, 0, 1))
        ax.set_title(f"{split_name} | GT mask", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Dataset visual QC — {split_name}", fontsize=14)
    plt.tight_layout()
    out = OUT_DIR / "figures" / f"dataset_visual_qc_{split_name}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

qc_paths = []
for split_name in ["train", "validation", "test", "external_inbreast"]:
    p = make_ground_truth_panel(split_name, n=SAVE_QUALITATIVE_N)
    if p is not None:
        qc_paths.append(p)
        print("Saved dataset visual QC:", p)
        display(IPyImage(filename=str(p)))


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        groups = min(groups, out_ch)
        while out_ch % groups != 0 and groups > 1:
            groups -= 1
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.e4 = ConvBlock(base*4, base*8)
        self.b = ConvBlock(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.g = nn.Conv2d(g_ch, inter_ch, 1)
        self.x = nn.Conv2d(x_ch, inter_ch, 1)
        self.psi = nn.Sequential(nn.SiLU(inplace=True), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())
    def forward(self, g, x):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return x * self.psi(self.g(g) + self.x(x))

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.e4 = ConvBlock(base*4, base*8)
        self.b = ConvBlock(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.a4 = AttentionGate(base*8, base*8, base*4)
        self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.a3 = AttentionGate(base*4, base*4, base*2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.a2 = AttentionGate(base*2, base*2, base)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.a1 = AttentionGate(base, base, max(base//2, 1))
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        u4 = self.u4(b); d4 = self.d4(torch.cat([u4, self.a4(u4, e4)], dim=1))
        u3 = self.u3(d4); d3 = self.d3(torch.cat([u3, self.a3(u3, e3)], dim=1))
        u2 = self.u2(d3); d2 = self.d2(torch.cat([u2, self.a2(u2, e2)], dim=1))
        u1 = self.u1(d2); d1 = self.d1(torch.cat([u1, self.a1(u1, e1)], dim=1))
        return self.out(d1)

class UpConv(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_AVAILABLE:
            raise RuntimeError("torchvision requis pour SwinTinyUNet")
        from torchvision.models import swin_t
        self.in_adapter = nn.Conv2d(1, 3, 1, bias=False)
        with torch.no_grad():
            self.in_adapter.weight.fill_(1.0)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.dec3 = UpConv(512, 384, 256)
        self.dec2 = UpConv(256, 192, 128)
        self.dec1 = UpConv(128, 96, 64)
        self.up0a = nn.ConvTranspose2d(64, 32, 2, 2)
        self.c0a = ConvBlock(32, 32)
        self.up0b = nn.ConvTranspose2d(32, 16, 2, 2)
        self.c0b = ConvBlock(16, 16)
        self.out = nn.Conv2d(16, out_ch, 1)
    def _nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96, 192, 384, 768]:
            return x.permute(0, 3, 1, 2).contiguous()
        return x
    def forward(self, x):
        y = self.in_adapter(x)
        y = (y - self.mean) / self.std
        feats = []
        for i, layer in enumerate(self.features):
            y = layer(y)
            if i in [1, 3, 5, 7]:
                feats.append(self._nchw(y))
        if len(feats) != 4:
            raise RuntimeError(f"Features Swin inattendues: {len(feats)}")
        s1, s2, s3, s4 = feats
        x = self.center(s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.c0a(self.up0a(x))
        x = self.c0b(self.up0b(x))
        if x.shape[-2:] != (IMAGE_SIZE, IMAGE_SIZE):
            x = F.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        return self.out(x)

def build_model(name):
    name = name.lower()
    if name == "unet":
        return UNet()
    if name == "attention_unet":
        return AttentionUNet()
    if name == "swin_tiny_unet":
        return SwinTinyUNet()
    raise ValueError(name)

def make_optimizer(name, model):
    if name == "swin_tiny_unet":
        enc, dec = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if n.startswith("features") or n.startswith("swin"):
                enc.append(p)
            else:
                dec.append(p)
        return torch.optim.AdamW(
            [{"params": enc, "lr": ENCODER_LR}, {"params": dec, "lr": BASE_LR}],
            weight_decay=WEIGHT_DECAY
        )
    return torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for name in MODELS_TO_RUN:
    try:
        m = build_model(name).to(device)
        with torch.no_grad():
            y = m(torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device))
        print(name, "params", count_params(m), "out", tuple(y.shape))
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print("[ERREUR modèle]", name, repr(e))
        if name != "swin_tiny_unet":
            raise

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)
        prob = torch.sigmoid(logits)
        dims = (1,2,3)
        inter = (prob * target).sum(dims)
        dice = (2*inter + self.smooth) / (prob.sum(dims) + target.sum(dims) + self.smooth)
        dice_loss = 1 - dice.mean()
        tp = (prob * target).sum(dims)
        fp = (prob * (1-target)).sum(dims)
        fn = ((1-prob) * target).sum(dims)
        tversky = (tp + self.smooth) / (tp + 0.3*fp + 0.7*fn + self.smooth)
        focal_tversky = torch.pow(1 - tversky, 0.75).mean()
        return 0.5*bce + 0.3*dice_loss + 0.2*focal_tversky

def binary_metrics(pred, target, eps=1e-7):
    pred = pred.astype(bool)
    target = target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    p = pred.sum()
    t = target.sum()
    dice = (2*tp + eps) / (p + t + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    iou = (tp + eps) / (tp + fp + fn + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps) if t > 0 else np.nan
    tn = np.logical_and(~pred, ~target).sum()
    return {
        "dice": float(dice), "iou": float(iou), "precision": float(precision), "recall": float(recall),
        "empty_pred": int(p == 0), "empty_target": int(t == 0),
        "pred_area": int(p), "target_area": int(t),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)
    }

def hd95_asd(pred, target):
    if not SCIPY_AVAILABLE:
        return {"hd95": np.nan, "asd": np.nan}
    pred = pred.astype(bool)
    target = target.astype(bool)
    if pred.sum() == 0 or target.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    pb = pred ^ binary_erosion(pred)
    tb = target ^ binary_erosion(target)
    if pb.sum() == 0 or tb.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    dt_t = distance_transform_edt(~tb)
    dt_p = distance_transform_edt(~pb)
    d = np.concatenate([dt_t[pb], dt_p[tb]]).astype(np.float32)
    return {"hd95": float(np.percentile(d, 95)), "asd": float(d.mean())}

In [ ]:
@torch.no_grad()
def collect_probs(model, loader, device):
    model.eval()
    rows, probs, masks = [], [], []
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        logits = model(x)
        pr = torch.sigmoid(logits).cpu().numpy()
        ma = batch["mask"].cpu().numpy()
        for i in range(x.shape[0]):
            rows.append({
                "sample_id": batch["sample_id"][i],
                "patient_id": batch["patient_id"][i],
                "dataset": batch["dataset"][i],
                "split": batch["split"][i],
            })
            probs.append(pr[i,0].astype(np.float32))
            masks.append(ma[i,0].astype(np.uint8))
    return rows, probs, masks

def evaluate_probs(rows, probs, masks, threshold, model_name, seed, split_eval, policy):
    out = []
    for row, prob, mask in zip(rows, probs, masks):
        pred = (prob >= threshold).astype(np.uint8)
        met = binary_metrics(pred, mask)
        surf = hd95_asd(pred, mask)
        out.append({
            **row, "model": model_name, "seed": seed, "split_eval": split_eval,
            "threshold": float(threshold), "threshold_policy": policy,
            **met, **surf
        })
    return pd.DataFrame(out)

def aggregate(df):
    result = {"n": len(df)}
    for c in ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred"]:
        result[c] = float(np.nanmean(df[c])) if c in df and len(df) else np.nan
    return result

def select_threshold(model_name, seed, val_rows, val_probs, val_masks):
    rows = []
    for th in THRESHOLDS:
        df = evaluate_probs(val_rows, val_probs, val_masks, th, model_name, seed, "validation", "candidate")
        rows.append({"threshold": th, **aggregate(df)})
    sweep = pd.DataFrame(rows)
    sweep["dist_to_05"] = (sweep["threshold"] - 0.5).abs()
    sweep = sweep.sort_values(["dice", "iou", "dist_to_05", "threshold"], ascending=[False, False, True, True])
    best = float(sweep.iloc[0]["threshold"])
    return best, sweep.sort_values("threshold")

In [ ]:
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / max(1, WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)
        with autocast(enabled=AMP and device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach().cpu()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)

@torch.no_grad()
def eval_loss(model, loader, criterion, device):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)
        with autocast(enabled=AMP and device.type == "cuda"):
            loss = criterion(model(x), y)
        total += float(loss.detach().cpu()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)

def save_examples(model_name, seed, split_name, rows, probs, masks, threshold, n=16):
    """Save an English qualitative segmentation panel.

    Columns per sample: input ROI, ground truth, prediction overlay.
    Green = ground truth, red = prediction, yellow = overlap.
    """
    if not MATPLOTLIB_AVAILABLE or len(rows) == 0:
        return None
    idxs = np.linspace(0, len(rows)-1, min(n, len(rows))).astype(int)
    n_rows = len(idxs)
    plt.figure(figsize=(12, max(3.0, n_rows * 3.0)))
    for r, idx in enumerate(idxs):
        sid = rows[idx]["sample_id"]
        rel = manifest.loc[manifest["sample_id"].astype(str).eq(str(sid)), "npz_path"].iloc[0]
        with np.load(DATA_ROOT / rel, allow_pickle=False) as z:
            img = np.squeeze(z["image"]).astype(np.float32)
        pred = (probs[idx] >= threshold).astype(np.uint8)
        mask = masks[idx].astype(np.uint8)

        overlay = np.stack([img, img, img], axis=-1)
        overlay[..., 0] = np.maximum(overlay[..., 0], pred * 1.0)
        overlay[..., 1] = np.maximum(overlay[..., 1], mask * 0.95)

        ax = plt.subplot(n_rows, 3, r*3 + 1)
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title("Input ROI", fontsize=9)
        ax.axis("off")

        ax = plt.subplot(n_rows, 3, r*3 + 2)
        ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
        ax.set_title("Ground truth mask", fontsize=9)
        ax.axis("off")

        ax = plt.subplot(n_rows, 3, r*3 + 3)
        ax.imshow(np.clip(overlay, 0, 1))
        ax.set_title("Prediction overlay", fontsize=9)
        ax.axis("off")

    plt.suptitle(
        f"Qualitative segmentation — {model_name} | seed {seed} | {split_name} | threshold={threshold:.3f}\n"
        "Green: ground truth, Red: prediction, Yellow: overlap",
        fontsize=13
    )
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    out = OUT_DIR / "figures" / f"{model_name}_seed{seed}_{split_name}_qualitative_segmentation.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

def run_experiment(model_name, seed):
    print(f"\n===== START {model_name} seed={seed} | {now()} =====")
    seed_everything(seed)
    loaders = make_loaders(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(model_name).to(device)
    optimizer = make_optimizer(model_name, model)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    criterion = CombinedLoss()
    scaler = GradScaler(enabled=AMP and device.type == "cuda")
    ckpt_path = OUT_DIR / "checkpoints" / f"{model_name}_seed{seed}_best.pt"

    best_val = float("inf")
    best_epoch = -1
    bad = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr = train_one_epoch(model, loaders["train"], optimizer, criterion, scaler, device)
        va = eval_loss(model, loaders["validation"], criterion, device)
        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        history.append({"model": model_name, "seed": seed, "epoch": epoch, "train_loss": tr, "val_loss": va, "lr": lr, "seconds": time.time()-t0})
        print(f"epoch {epoch:03d} train={tr:.5f} val={va:.5f} lr={lr:.2e}")

        if va < best_val - 1e-5:
            best_val = va
            best_epoch = epoch
            bad = 0
            torch.save({"model": model_name, "seed": seed, "epoch": epoch, "state_dict": model.state_dict()}, ckpt_path)
        else:
            bad += 1

        pd.DataFrame(history).to_csv(OUT_DIR / "metrics" / f"{model_name}_seed{seed}_history.csv", index=False)
        if bad >= PATIENCE:
            print("Early stopping. Best epoch:", best_epoch, "best validation loss:", best_val)
            break

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])

    caches = {}
    for split in ["train_eval", "validation", "test", "external_inbreast"]:
        if len(loaders[split].dataset) == 0:
            caches[split] = ([], [], [])
        else:
            caches[split] = collect_probs(model, loaders[split], device)

    val_rows, val_probs, val_masks = caches["validation"]
    selected_th, sweep = select_threshold(model_name, seed, val_rows, val_probs, val_masks)
    sweep["model"] = model_name
    sweep["seed"] = seed
    sweep.to_csv(OUT_DIR / "metrics" / f"{model_name}_seed{seed}_threshold_sweep_validation.csv", index=False)

    detailed_parts = []
    for split in ["train_eval", "validation", "test", "external_inbreast"]:
        rows, probs, masks = caches[split]
        if len(rows) == 0:
            continue
        split_label = "train" if split == "train_eval" else split
        for th, policy in [(selected_th, "selected_on_cbis_validation"), (0.50, "fixed_0_50")]:
            detailed_parts.append(evaluate_probs(rows, probs, masks, th, model_name, seed, split_label, policy))
        fig_path = save_examples(model_name, seed, split_label, rows, probs, masks, selected_th)
        if fig_path is not None:
            print("Saved qualitative segmentation panel:", fig_path)

    detailed = pd.concat(detailed_parts, ignore_index=True)
    detailed.to_csv(OUT_DIR / "metrics" / f"{model_name}_seed{seed}_detailed_metrics.csv", index=False)

    summaries = []
    for (split, policy), g in detailed.groupby(["split_eval", "threshold_policy"]):
        summaries.append({
            "model": model_name, "seed": seed, "split": split, "threshold_policy": policy,
            "selected_threshold": selected_th if policy == "selected_on_cbis_validation" else 0.50,
            "best_epoch": best_epoch, "best_val_loss": best_val,
            **aggregate(g)
        })
    summary = pd.DataFrame(summaries)
    summary.to_csv(OUT_DIR / "metrics" / f"{model_name}_seed{seed}_summary.csv", index=False)
    print(f"===== END {model_name} seed={seed} | selected threshold={selected_th} =====")
    return summary, sweep

In [ ]:
all_summaries, all_sweeps, failures = [], [], []

for model_name in MODELS_TO_RUN:
    for seed in TRAINING_SEEDS:
        try:
            summary, sweep = run_experiment(model_name, seed)
            all_summaries.append(summary)
            all_sweeps.append(sweep)
        except Exception as e:
            print("[FAIL]", model_name, seed, repr(e))
            failures.append({"model": model_name, "seed": seed, "error": repr(e)})
            pd.DataFrame(failures).to_csv(OUT_DIR / "logs" / "experiment_failures.csv", index=False)
            continue

if not all_summaries:
    raise RuntimeError("No experiment completed successfully")

raw = pd.concat(all_summaries, ignore_index=True)
raw.to_csv(OUT_DIR / "metrics" / "roi256_experiments_summary_raw_by_seed.csv", index=False)
display(raw)

if all_sweeps:
    pd.concat(all_sweeps, ignore_index=True).to_csv(OUT_DIR / "metrics" / "roi256_threshold_sweeps_validation_all.csv", index=False)

if failures:
    display(pd.DataFrame(failures))
else:
    print("All requested experiments have completed successfully.")

In [ ]:
raw = pd.read_csv(OUT_DIR / "metrics" / "roi256_experiments_summary_raw_by_seed.csv")

metrics_cols = ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred"]
rows = []
for (model, split, policy), g in raw.groupby(["model", "split", "threshold_policy"]):
    row = {
        "model": model,
        "split": split,
        "threshold_policy": policy,
        "n_seeds": g["seed"].nunique(),
        "threshold_mean": g["selected_threshold"].mean(),
        "threshold_std": g["selected_threshold"].std(ddof=1) if len(g) > 1 else 0.0,
    }
    for c in metrics_cols:
        row[f"{c}_mean"] = g[c].mean()
        row[f"{c}_std"] = g[c].std(ddof=1) if len(g) > 1 else 0.0
    rows.append(row)

mean_std = pd.DataFrame(rows).sort_values(["split", "threshold_policy", "dice_mean"], ascending=[True, True, False])
mean_std.to_csv(OUT_DIR / "metrics" / "roi256_experiments_summary_mean_std.csv", index=False)
try:
    mean_std.to_excel(OUT_DIR / "metrics" / "roi256_experiments_summary_mean_std.xlsx", index=False)
except Exception as e:
    print("Excel export failed:", repr(e))

display(mean_std)

article = mean_std.copy()
for c in ["dice", "iou", "precision", "recall", "hd95", "asd"]:
    article[c] = article.apply(lambda r: f"{r[c + '_mean']:.3f} ± {r[c + '_std']:.3f}", axis=1)
article["threshold"] = article.apply(lambda r: f"{r['threshold_mean']:.3f} ± {r['threshold_std']:.3f}", axis=1)
article = article[["model", "split", "threshold_policy", "n_seeds", "threshold", "dice", "iou", "precision", "recall", "hd95", "asd"]]
article.to_csv(OUT_DIR / "metrics" / "roi256_article_table.csv", index=False)
display(article)
print("Article-ready metric tables saved in:", OUT_DIR / "metrics")


In [ ]:
def build_confusion_tables():
    detail_files = sorted((OUT_DIR / "metrics").glob("*_detailed_metrics.csv"))
    if not detail_files:
        print("No detailed metric files were found. Run the experiments first.")
        return None, None

    detailed = pd.concat([pd.read_csv(p) for p in detail_files], ignore_index=True)
    required = {"model", "seed", "split_eval", "threshold_policy", "tp", "fp", "fn", "tn"}
    missing = required - set(detailed.columns)
    if missing:
        raise ValueError(f"Detailed metrics do not contain confusion-count columns: {missing}")

    by_seed = detailed.groupby(["model", "seed", "split_eval", "threshold_policy"], as_index=False)[["tn", "fp", "fn", "tp"]].sum()
    by_seed["pixel_accuracy"] = (by_seed["tp"] + by_seed["tn"]) / (by_seed["tp"] + by_seed["tn"] + by_seed["fp"] + by_seed["fn"]).clip(lower=1)
    by_seed["pixel_sensitivity"] = by_seed["tp"] / (by_seed["tp"] + by_seed["fn"]).clip(lower=1)
    by_seed["pixel_specificity"] = by_seed["tn"] / (by_seed["tn"] + by_seed["fp"]).clip(lower=1)

    agg = by_seed.groupby(["model", "split_eval", "threshold_policy"], as_index=False)[["tn", "fp", "fn", "tp"]].sum()
    agg["pixel_accuracy"] = (agg["tp"] + agg["tn"]) / (agg["tp"] + agg["tn"] + agg["fp"] + agg["fn"]).clip(lower=1)
    agg["pixel_sensitivity"] = agg["tp"] / (agg["tp"] + agg["fn"]).clip(lower=1)
    agg["pixel_specificity"] = agg["tn"] / (agg["tn"] + agg["fp"]).clip(lower=1)

    by_seed.to_csv(OUT_DIR / "metrics" / "roi256_pixel_confusion_matrices_by_seed.csv", index=False)
    agg.to_csv(OUT_DIR / "metrics" / "roi256_pixel_confusion_matrices_aggregated.csv", index=False)
    return by_seed, agg

def plot_confusion_matrix_row(row):
    if not MATPLOTLIB_AVAILABLE:
        return None
    cm = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=np.float64)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)

    plt.figure(figsize=(5.5, 4.8))
    plt.imshow(cm_norm, vmin=0, vmax=1)
    plt.xticks([0, 1], ["Predicted negative", "Predicted positive"], rotation=25, ha="right")
    plt.yticks([0, 1], ["Ground-truth negative", "Ground-truth positive"])
    title = f"Pixel-level confusion matrix\n{row['model']} | {row['split_eval']} | {row['threshold_policy']}"
    plt.title(title)
    for i in range(2):
        for j in range(2):
            txt = f"{int(cm[i, j]):,}\n{cm_norm[i, j]*100:.2f}%"
            plt.text(j, i, txt, ha="center", va="center")
    plt.colorbar(label="Row-normalized proportion")
    plt.tight_layout()
    safe = f"{row['model']}_{row['split_eval']}_{row['threshold_policy']}".replace("/", "_").replace(" ", "_")
    out = OUT_DIR / "figures" / f"confusion_matrix_{safe}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

by_seed_cm, agg_cm = build_confusion_tables()
if agg_cm is not None:
    print("Pixel-level confusion matrices by seed:")
    display(by_seed_cm.head(20))
    print("Aggregated pixel-level confusion matrices:")
    display(agg_cm)

    cm_paths = []
    for _, row in agg_cm.iterrows():
        p = plot_confusion_matrix_row(row)
        if p is not None:
            cm_paths.append(p)
    print(f"Saved {len(cm_paths)} confusion-matrix figures in:", OUT_DIR / "figures")

    priority = agg_cm[agg_cm["threshold_policy"].eq("selected_on_cbis_validation")].copy()
    for _, row in priority.iterrows():
        p = OUT_DIR / "figures" / f"confusion_matrix_{row['model']}_{row['split_eval']}_{row['threshold_policy']}.png"
        if p.exists():
            display(IPyImage(filename=str(p)))


In [ ]:
if MATPLOTLIB_AVAILABLE:
    for hp in sorted((OUT_DIR / "metrics").glob("*_history.csv")):
        h = pd.read_csv(hp)
        if h.empty:
            continue
        model = h["model"].iloc[0]
        seed = h["seed"].iloc[0]
        plt.figure(figsize=(7,5))
        plt.plot(h["epoch"], h["train_loss"], label="train")
        plt.plot(h["epoch"], h["val_loss"], label="validation")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"{model} seed {seed}")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "figures" / f"{model}_seed{seed}_loss_curve.png", dpi=150)
        plt.close()

    sweep_all = OUT_DIR / "metrics" / "roi256_threshold_sweeps_validation_all.csv"
    if sweep_all.exists():
        sweeps = pd.read_csv(sweep_all)
        for (model, seed), g in sweeps.groupby(["model", "seed"]):
            plt.figure(figsize=(7,5))
            plt.plot(g["threshold"], g["dice"], marker="o", label="Dice")
            plt.plot(g["threshold"], g["iou"], marker="o", label="IoU")
            plt.xlabel("Threshold")
            plt.ylabel("Metric")
            plt.title(f"Validation threshold sweep — {model} seed {seed}")
            plt.legend()
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(OUT_DIR / "figures" / f"{model}_seed{seed}_threshold_sweep.png", dpi=150)
            plt.close()

print("Figures saved in:", OUT_DIR / "figures")

In [ ]:
counts_md = manifest.groupby(["dataset", "split"]).size().reset_index(name="n").to_markdown(index=False)

card = f'''
# ROI 256 Training Results — CBIS-DDSM / INbreast

Generated: {now()}

## Dataset

Root: `{DATA_ROOT}`

This experiment uses ROI oracle crops of size 256x256 generated from full-field 448x448 CBIS-DDSM and INbreast samples.

## Experimental governance

- Training split: CBIS-DDSM train only.
- Threshold selection: CBIS-DDSM validation only.
- Internal test: CBIS-DDSM test, sealed.
- External validation: INbreast external_inbreast, sealed.
- No INbreast sample is used for training, threshold tuning, checkpoint selection, or hyperparameter tuning.
- Evaluation is ROI/oracle-crop segmentation and assumes prior lesion localization.

## Data counts

{counts_md}

## Configuration

- models: {MODELS_TO_RUN}
- seeds: {TRAINING_SEEDS}
- image size: {IMAGE_SIZE}
- batch size: {BATCH_SIZE}
- epochs: {EPOCHS}
- patience: {PATIENCE}
- thresholds: {THRESHOLDS}
- loss: BCEWithLogits + soft Dice + Focal Tversky
- optimizer: AdamW
- scheduler: warmup + cosine decay
- AMP: {AMP}
- augmentation train only: {AUGMENT_TRAIN}

## Main outputs

- `metrics/roi256_experiments_summary_raw_by_seed.csv`
- `metrics/roi256_experiments_summary_mean_std.csv`
- `metrics/roi256_experiments_summary_mean_std.xlsx`
- `metrics/roi256_article_table.csv`
- `metrics/*_threshold_sweep_validation.csv`
- `metrics/roi256_pixel_confusion_matrices_by_seed.csv`
- `metrics/roi256_pixel_confusion_matrices_aggregated.csv`
- `checkpoints/*_best.pt`
- `figures/dataset_visual_qc_*.png`
- `figures/*_qualitative_segmentation.png`
- `figures/confusion_matrix_*.png`
- `figures/*_loss_curve.png`
'''
(OUT_DIR / "ROI256_EXPERIMENT_CARD.md").write_text(card, encoding="utf-8")
print(card)

In [ ]:
zip_base = Path("/kaggle/working/ROI256_TRAINING_RESULTS")
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
sha = sha256_file(zip_path)
pd.DataFrame([{"zip_path": str(zip_path), "sha256": sha}]).to_csv("/kaggle/working/ROI256_TRAINING_RESULTS_sha256.csv", index=False)

print("Final archive:", zip_path)
print("SHA256:", sha)
print("Key files:")
for p in [
    OUT_DIR / "ROI256_EXPERIMENT_CARD.md",
    OUT_DIR / "metrics" / "roi256_experiments_summary_raw_by_seed.csv",
    OUT_DIR / "metrics" / "roi256_experiments_summary_mean_std.csv",
    OUT_DIR / "metrics" / "roi256_article_table.csv",
]:
    print(" -", p, "exists=", p.exists())